# Генеративно-состязательные сети

**Проект «ИИ для учёных».** Практический блокнот к странице алгоритма «Генеративно-состязательные сети».

## Научная цель

Генеративно-состязательная сеть (GAN) состоит из двух нейронных сетей, обучаемых в соревновании. Генератор создаёт новые объекты, а дискриминатор пытается отличить их от настоящих. Генератор постепенно учится обманывать дискриминатор, и его выборки становятся всё правдоподобнее. Для учёного это способ получить синтетические данные или сгенерировать кандидатов для проверки.

В этом блокноте мы обучим компактный GAN воспроизводить нетривиальное двумерное распределение — «два полумесяца». Двумерные данные позволяют наглядно увидеть и обучение, и типичные проблемы (коллапс мод). Блокнот исполняется на CPU за минуту.

**Важное предупреждение.** Синтетические данные GAN могут содержать артефакты и придуманные структуры; трактовать их как фактические измерения нельзя.

## Интуиция за методом

Представьте соревнование между изготовителем подделок и экспертом-детектором. Изготовитель раз за разом пытается сделать копию, неотличимую от оригинала. Эксперт тренируется замечать малейшие несоответствия. Чем лучше становится детектор, тем тщательнее работает изготовитель — и наоборот. Эта гонка продолжается до тех пор, пока подделки не станут неотличимы от оригиналов.

Именно так обучается генеративно-состязательная сеть:

- **Генератор** (generator) — принимает случайный шум и превращает его в объект-кандидат (изображение, набор чисел, сигнал). Это «изготовитель подделок».
- **Дискриминатор** (discriminator) — получает либо настоящий объект из данных, либо кандидат от генератора, и выдаёт вероятность того, что объект настоящий. Это «эксперт-детектор».

Обе сети обучаются одновременно и противоположным образом:
- Дискриминатор учится различать настоящее и сгенерированное.
- Генератор учится делать свои объекты настолько похожими на настоящие, чтобы дискриминатор ошибался.

После обучения дискриминатор больше не нужен — используется только генератор.

Терминологический минимум для этого блокнота:
- **Вектор шума** — случайный числовой вектор, из которого генератор создаёт объект. Каждый новый шум → новый объект.
- **Минимаксная игра** — математическое описание соревнования: генератор минимизирует свою «заметность», дискриминатор максимизирует способность различать.
- **Функция потерь BCE** (Binary Cross-Entropy) — стандартная функция потерь для задачи «настоящее или нет».
- **Коллапс мод** (mode collapse) — патология: генератор научился обманывать дискриминатор, но делает только однотипные объекты, игнорируя разнообразие данных.
- **Эпоха / шаг обучения** — здесь используем «шаги»: каждый шаг = одно обновление весов каждой сети на одном пакете данных.

## 1. Установка библиотек

В Google Colab большинство пакетов уже установлено. Ячейка ниже гарантирует наличие нужных библиотек. При локальном запуске она также корректна.

In [ ]:
%pip install -q torch==2.3.1 scikit-learn==1.5.1 numpy==1.26.4 matplotlib==3.9.2

## 2. Единый стиль графиков

Графики оформляются в едином визуальном стиле сайта «ИИ для учёных». Ниже встроено содержимое стилевого шаблона `notebooks/viz_style.py`.

In [ ]:
# Единый стилевой шаблон графиков проекта «ИИ для учёных».
# Значения синхронизированы с токенами темы сайта (styles/tokens.css).
VIZ_TOKENS = {
    "background": "#ffffff",
    "node_fill":  "#eef1f6",
    "node_text":  "#0e1726",
    "edge":       "#4d5e78",
    "grid":       "#dce2ec",
    "series":     ["#2563eb", "#0d9488", "#b45309", "#4d5e78"],
}
VIZ = VIZ_TOKENS


def apply_viz_style():
    """Настраивает matplotlib под единый стиль сайта."""
    import matplotlib as mpl
    from cycler import cycler
    t = VIZ_TOKENS
    mpl.rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Segoe UI", "DejaVu Sans", "Arial", "Helvetica"],
        "font.size": 12, "axes.titlesize": 15, "axes.titleweight": "semibold",
        "axes.labelsize": 13, "xtick.labelsize": 11, "ytick.labelsize": 11,
        "legend.fontsize": 11,
        "figure.facecolor": t["background"], "axes.facecolor": t["background"],
        "savefig.facecolor": t["background"], "figure.dpi": 110, "savefig.dpi": 150,
        "axes.prop_cycle": cycler(color=t["series"]),
        "text.color": t["node_text"], "axes.labelcolor": t["node_text"],
        "axes.titlecolor": t["node_text"], "axes.edgecolor": t["edge"],
        "xtick.color": t["edge"], "ytick.color": t["edge"], "axes.linewidth": 1.2,
        "axes.grid": True, "grid.color": t["grid"], "grid.linewidth": 1.0,
        "grid.linestyle": "-", "axes.axisbelow": True,
        "lines.linewidth": 2.0, "lines.markersize": 6, "patch.linewidth": 1.5,
        "axes.spines.top": False, "axes.spines.right": False,
        "legend.frameon": False,
    })


def get_palette(n=None):
    """Возвращает список цветов рядов из токенов темы."""
    series = VIZ_TOKENS["series"]
    if n is None:
        return list(series)
    return [series[i % len(series)] for i in range(n)]


apply_viz_style()
print("Единый стиль графиков подключён.")

## 3. Демонстрационные данные

Используем классический двумерный набор `make_moons` из `scikit-learn` — две переплетённые дуги-«полумесяца». Это удобный эталон: распределение нетривиальное, но двумерное, поэтому и настоящие, и сгенерированные точки можно показать на одном графике.

In [ ]:
import numpy as np
import torch
from sklearn.datasets import make_moons

torch.manual_seed(0)
np.random.seed(0)

X, _ = make_moons(n_samples=2000, noise=0.07, random_state=0)
X = X.astype('float32')
# Нормировка: нулевое среднее, единичный разброс.
X = (X - X.mean(0)) / X.std(0)
real = torch.from_numpy(X)
print(f'Настоящих точек: {len(X)}, размерность: {X.shape[1]}')

## 4. Применение метода

### Шаг 1. Архитектура генератора и дискриминатора

**Генератор:** принимает вектор случайного шума размерности `NOISE = 8` и превращает его в двумерную точку-кандидат. Архитектура простая — два скрытых слоя с функцией активации ReLU.

**Дискриминатор:** принимает двумерную точку и выдаёт вероятность (от 0 до 1): 1 — «уверен, что настоящая», 0 — «уверен, что сгенерированная». Использует `LeakyReLU` вместо `ReLU` — это лучше работает в дискриминаторах, так как позволяет небольшим отрицательным значениям проходить через функцию активации.

Задача на двух полумесяцах выбрана специально: это нетривиальное нелинейное распределение, которое наглядно показывает возможности и ограничения GAN.

In [ ]:
import torch.nn as nn

NOISE = 8          # размерность входного шума генератора


class Generator(nn.Module):
    """Превращает вектор шума в двумерную точку-кандидат."""

    def __init__(self, noise=NOISE, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(noise, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 2),
        )

    def forward(self, z):
        return self.net(z)


class Discriminator(nn.Module):
    """Оценивает вероятность того, что точка настоящая."""

    def __init__(self, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, hidden), nn.LeakyReLU(0.2),
            nn.Linear(hidden, 1), nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


gen = Generator()
disc = Discriminator()
print('Генератор и дискриминатор собраны.')

### Шаг 2. Состязательное обучение

Цикл обучения делает на каждом шаге два обновления:

1. **Шаг дискриминатора:** берём пакет настоящих точек и пакет сгенерированных. Дискриминатор должен выдавать 1 для настоящих и 0 для сгенерированных. Считаем потерю и обновляем только веса дискриминатора.

2. **Шаг генератора:** снова генерируем точки. Теперь дискриминатор уже обновлён — нам нужно обмануть его: заставить выдать 1 на наши сгенерированные точки. Считаем потерю и обновляем только веса генератора.

Важно: `.detach()` при шаге дискриминатора отсекает граф вычислений от генератора — иначе обновление дискриминатора случайно изменило бы и генератор.

В снимки `snapshots` сохраняем координаты сгенерированных точек на шагах 250, 800 и 2500 — чтобы потом показать эволюцию.

In [ ]:
opt_g = torch.optim.Adam(gen.parameters(), lr=2e-3, betas=(0.5, 0.9))
opt_d = torch.optim.Adam(disc.parameters(), lr=2e-3, betas=(0.5, 0.9))
bce = nn.BCELoss()

BATCH = 128
n_steps = 2500
history = {'d': [], 'g': []}
snapshots = {}

for step in range(1, n_steps + 1):
    idx = torch.randint(0, len(real), (BATCH,))
    real_batch = real[idx]

    # --- Шаг дискриминатора: отличить настоящие от сгенерированных ---
    z = torch.randn(BATCH, NOISE)
    fake = gen(z).detach()
    d_real = disc(real_batch)
    d_fake = disc(fake)
    loss_d = bce(d_real, torch.ones(BATCH)) + \
             bce(d_fake, torch.zeros(BATCH))
    opt_d.zero_grad(); loss_d.backward(); opt_d.step()

    # --- Шаг генератора: обмануть дискриминатор ---
    z = torch.randn(BATCH, NOISE)
    gen_out = gen(z)
    loss_g = bce(disc(gen_out), torch.ones(BATCH))
    opt_g.zero_grad(); loss_g.backward(); opt_g.step()

    history['d'].append(loss_d.item())
    history['g'].append(loss_g.item())
    if step in (250, 800, n_steps):
        with torch.no_grad():
            snapshots[step] = gen(torch.randn(800, NOISE)).numpy()
    if step % 500 == 0:
        print(f'Шаг {step:4d}: потеря D {loss_d.item():.3f}, потеря G {loss_g.item():.3f}')

### Динамика обучения и качество выборок

Слева — потери двух сетей по шагам. Справа — снимки сгенерированного распределения на разных этапах обучения поверх настоящих данных.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.4))

# Сглаженные кривые потерь.
def smooth(v, k=25):
    v = np.asarray(v)
    ker = np.ones(k) / k
    return np.convolve(v, ker, mode='valid')

axes[0].plot(smooth(history['d']), color=VIZ['series'][0],
             label='Дискриминатор')
axes[0].plot(smooth(history['g']), color=VIZ['series'][2],
             label='Генератор')
axes[0].set_title('Потери в состязательном обучении')
axes[0].set_xlabel('Шаг обучения')
axes[0].set_ylabel('Функция потерь (сглажено)')
axes[0].legend(loc='upper right')

axes[1].scatter(X[:, 0], X[:, 1], s=12, color=VIZ['series'][3],
                alpha=0.35, label='Настоящие данные')
final = snapshots[n_steps]
axes[1].scatter(final[:, 0], final[:, 1], s=14,
                color=VIZ['series'][1], alpha=0.7,
                label='Генерация GAN')
axes[1].set_title('Итоговое распределение генератора')
axes[1].set_xlabel('Признак 1')
axes[1].set_ylabel('Признак 2')
axes[1].legend(loc='upper right')

fig.tight_layout()
plt.show()

**Как читать эти графики:**

- **Левый — кривые потерь:** в отличие от обычных задач обучения, здесь потери не обязаны монотонно убывать. Сети конкурируют, и равновесие — это колебание потерь около некоторого среднего. Если потеря генератора резко растёт, а потеря дискриминатора падает до нуля — это признак срыва обучения. Если потеря дискриминатора всегда около 0 — он «победил» и генератор не учится.
- **Правый — финальное распределение:** серые точки — настоящие данные (два полумесяца), цветные — сгенерированные. При хорошем обучении два облака точек совпадают по форме и плотности. Если сгенерированные точки образуют один комок — это коллапс мод.

In [ ]:
# Эволюция генератора: распределение выборок на трёх этапах обучения.
steps_show = sorted(snapshots)
fig, axes = plt.subplots(1, len(steps_show), figsize=(13.5, 4.4),
                         sharex=True, sharey=True)
for ax, st in zip(axes, steps_show):
    ax.scatter(X[:, 0], X[:, 1], s=8, color=VIZ['series'][3],
               alpha=0.25)
    pts = snapshots[st]
    ax.scatter(pts[:, 0], pts[:, 1], s=10, color=VIZ['series'][1],
               alpha=0.7)
    ax.set_title(f'Шаг обучения {st}')
    ax.set_xlabel('Признак 1')
axes[0].set_ylabel('Признак 2')
fig.suptitle('Как генератор постепенно осваивает форму данных',
             fontsize=13)
fig.tight_layout()
plt.show()

**Как читать этот график (эволюция генератора):**

Три панели показывают снимки сгенерированных точек на разных этапах обучения (шаги 250, 800, 2500). Серые точки — настоящие данные, цветные — генерация на данном шаге.

- **Ранний этап (250 шагов):** генератор только начал — точки сгруппированы в случайное облако без формы.
- **Середина обучения (800 шагов):** видно, что генератор «нащупывает» структуру — облако начинает изгибаться.
- **Конец обучения (2500 шагов):** генератор воспроизводит оба полумесяца. Если один полумесяц покрыт хуже — это начало коллапса мод.

## 5. Интерпретация результата

- **Эволюция выборок**: на ранних шагах точки генератора собраны в комок, к концу обучения они повторяют форму двух полумесяцев. Так видно, как состязание доводит выборки до правдоподобия.
- **Итоговое распределение**: облако генерации должно совпадать с облаком настоящих данных по форме и плотности.
- **Кривые потерь** в GAN не убывают монотонно: сети соревнуются, и потери колеблются около равновесия. Резкое расхождение означает срыв обучения.
- **Коллапс мод**: если генератор покрывает лишь часть данных (например, один полумесяц), он потерял разнообразие. Это типичная проблема GAN.

У GAN нет встроенной метрики правдоподобия — качество оценивают косвенно. Синтетические данные нельзя использовать как фактические измерения.

## 6. Попробуйте на своих данных

Эта учебная схема подходит для генерации векторных данных невысокой размерности.

1. Подготовьте матрицу `X` формы `(наблюдение, признак)` и нормируйте признаки (нулевое среднее, единичный разброс).
2. Снимите комментарии в ячейке ниже, подставьте свои данные и при необходимости измените размерность выхода генератора и входа дискриминатора (`2` -> число признаков).
3. Следите за коллапсом мод: сравнивайте разнообразие генерации с исходными данными.
4. Для изображений берут свёрточные генератор и дискриминатор (архитектура DCGAN); для устойчивости применяют потерю Вассерштейна.

## 7. Поэкспериментируйте сами

Попробуйте изменить один параметр и перезапустить обучение с нуля (`torch.manual_seed(0)`):

| Параметр | Что изменить | Что наблюдать |
|---|---|---|
| `n_steps = 1000` | Сократить до 1000 шагов | Успевает ли генератор воспроизвести оба полумесяца? |
| `NOISE = 2` | Уменьшить шум до 2 | Вектор шума слишком мал — труднее генерировать разнообразие |
| `lr = 1e-3` (только генератор) | Замедлить генератор | Что происходит, когда дискриминатор «опережает»? |
| `lr = 1e-3` (только дискриминатор) | Замедлить дискриминатор | Что происходит, когда генератор «убегает» вперёд? |
| `noise = 0.2` в `make_moons` | Добавить больше шума к данным | Сложнее ли GAN воспроизводить менее чёткое распределение? |

Подсказка по диагностике коллапса мод: если в финальном графике все сгенерированные точки сосредоточены лишь в одном полумесяце — генератор потерял разнообразие. Попробуйте уменьшить `lr` дискриминатора.

In [ ]:
# --- Шаблон загрузки своих векторных данных ---
# import pandas as pd
#
# X = pd.read_csv('my_data.csv').to_numpy(dtype='float32')
# X = (X - X.mean(0)) / (X.std(0) + 1e-8)
# real = torch.from_numpy(X)
#
# n_features = X.shape[1]
# print(f'Объектов: {len(X)}, признаков: {n_features}')
# В Generator и Discriminator замените размерность 2 на n_features.

## 8. Краткие выводы

- GAN — это состязание двух сетей: генератор создаёт объекты-кандидаты, дискриминатор учится их разоблачать. Эта гонка доводит генерацию до правдоподобия.
- Потери обеих сетей не убывают монотонно — они колеблются около равновесия. Это нормально.
- Коллапс мод — ключевая проблема: генератор начинает делать однотипные объекты. Следите за разнообразием выборок.
- Синтетические данные GAN могут содержать артефакты и не подходят для количественных научных выводов. Их используют для аугментации и предобработки с последующей проверкой.
- Для более устойчивого обучения применяют потерю Вассерштейна (WGAN-GP) и спектральную нормализацию.
- По качеству выборок GAN сегодня во многом уступает диффузионным моделям, но выигрывает в скорости генерации после обучения.

## Три вопроса для самопроверки

Ответьте до того, как раскроете подсказку, — это проверяет, что метод понят, а не просто запущен.

1. На графике «Итоговое распределение генератора» сгенерированные точки плотно покрывают один полумесяц и почти не попадают во второй. Как называется это явление, и что оно говорит о качестве обученного генератора?
2. Вы хотите применить GAN к собственным экспериментальным данным о молекулярных дескрипторах (100 признаков, 5 000 образцов). Какие две архитектурные настройки нужно изменить в первую очередь по сравнению с демонстрационным примером, и почему?
3. Функция потерь дискриминатора с первых же шагов упала до нуля и больше не меняется, а функция потерь генератора монотонно растёт. Что произошло и какой параметр обучения стоит скорректировать в первую очередь?

<details>
<summary>Показать ориентиры для ответов</summary>

1. Это коллапс мод (mode collapse): генератор нашёл одно правдоподобное состояние и сосредоточился на нём, перестав воспроизводить полное разнообразие данных. Генерация технически «обманывает» дискриминатор, но не отражает исходное распределение.
2. Следует изменить размерность выхода генератора и входа дискриминатора с 2 на 100, а также, вероятно, расширить скрытые слои (hidden), поскольку 100-мерное пространство требует большей ёмкости сети, чем двумерное.
3. Дискриминатор «победил»: он научился безошибочно отличать настоящие данные от сгенерированных, и его градиент перестал нести полезный сигнал для генератора. Первый шаг — снизить скорость обучения дискриминатора относительно генератора или перейти на потерю Вассерштейна (WGAN), которая даёт более устойчивый градиент.
</details>
